# 1. Cleaning Fact tables

## 1.1. Cleaning sales_orders

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_orders = spark.readStream.table("azuresalesdatabricks.bronze.sales_orders")

def clean_and_merge_orders(batch_df, batch_id):
    cleaned = (batch_df
        # Fix 1: two date formats in the source (yyyy-MM-dd and MM/dd/yyyy) -> one real DATE type
        .withColumn("order_date_clean",
            F.coalesce(
                F.expr("try_to_date(order_date, 'yyyy-MM-dd')"),
                F.expr("try_to_date(order_date, 'MM/dd/yyyy')")
            ))
        # Fix 2: inconsistent casing/whitespace ("  Completed  ", "REFUNDED") -> one clean form
        .withColumn("order_status_clean", F.initcap(F.trim(F.col("order_status"))))
        # Fix 3: blank string customer_id (walk-in sales) -> a real NULL, not an empty string
        .withColumn("customer_id_clean",
            F.when(F.trim(F.col("customer_id")) == "", None).otherwise(F.col("customer_id")))
    )

    # Fix 4: duplicate order_id rows within the same file -> keep only the most recent
    w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())
    deduped = (cleaned
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn"))

    final = deduped.select(
        "order_id",
        F.col("order_date_clean").alias("order_date"),
        F.col("customer_id_clean").alias("customer_id"),
        "store_id", "rep_id",
        F.col("order_status_clean").alias("order_status"),
        "payment_method", "order_total",
        "_source_file", "_ingested_at"
    )

    final.createOrReplaceTempView("orders_batch")
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.silver.sales_orders AS target
        USING orders_batch AS source
        ON target.order_id = source.order_id
        WHEN NOT MATCHED THEN INSERT *
    """)

query = (bronze_orders.writeStream
    .foreachBatch(clean_and_merge_orders)
    .option("checkpointLocation", "abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/sales_orders/")
    .trigger(availableNow=True)
    .start())
query.awaitTermination()

## 1.2. Cleaning order_items

In [0]:
# ============================================================
# STEP 1 — Set up the streaming source
# ============================================================
# Read from bronze.order_items as a STREAM, not a static table.
# This means: only rows that arrived in bronze since the last time
# this checkpoint ran will show up here. Old rows we already
# processed are automatically skipped — that's the whole point
# of using readStream instead of spark.read.
bronze_items = spark.readStream.table("azuresalesdatabricks.bronze.order_items")


# ============================================================
# STEP 2 — Define what happens to EACH micro-batch of new rows
# ============================================================
# foreachBatch doesn't run this function once — it runs it once
# PER BATCH of new data that shows up. batch_df = the new rows,
# batch_id = a sequential number Spark assigns automatically.
def clean_and_merge_items(batch_df, batch_id):

    # --------------------------------------------------------
    # 2a. Load the list of order_ids we know are REAL
    # --------------------------------------------------------
    # We check against silver.sales_orders (not bronze!) because
    # silver is already cleaned — we want to validate against
    # trustworthy data, not raw/dirty data.
    valid_order_ids = spark.table("azuresalesdatabricks.silver.sales_orders").select("order_id")

    # --------------------------------------------------------
    # 2b. Tag every row: is it good, or bad — and WHY
    # --------------------------------------------------------
    tagged = (batch_df
        # Fix data type: quantity arrived as a string from CSV,
        # cast it to a real integer so we can compare it numerically
        .withColumn("quantity", F.col("quantity").cast("int"))

        # LEFT JOIN against valid_order_ids:
        # - order_items is the LEFT table -> every row survives, no matter what
        # - if a row's order_id has no match in sales_orders,
        #   valid_order_id comes back as NULL for that row
        .join(valid_order_ids.withColumnRenamed("order_id", "valid_order_id"),
              batch_df.order_id == F.col("valid_order_id"), "left")

        # Now stamp a reason on any row that's broken.
        # This checks conditions TOP TO BOTTOM, first match wins:
        .withColumn("_quarantine_reason",
            F.when(F.col("quantity") <= 0, F.lit("invalid_quantity"))       # bad quantity?
             .when(F.col("valid_order_id").isNull(), F.lit("orphan_order_id"))  # no matching order?
             .otherwise(F.lit(None)))  # neither problem -> row is clean, reason = NULL

        # We only needed valid_order_id to do the check above —
        # drop it now so it doesn't pollute our final table's schema
        .drop("valid_order_id")
    )

    # --------------------------------------------------------
    # 2c. Split into two separate DataFrames: good vs bad
    # --------------------------------------------------------
    good = tagged.filter(F.col("_quarantine_reason").isNull()).drop("_quarantine_reason")
    bad  = tagged.filter(F.col("_quarantine_reason").isNotNull())

    # --------------------------------------------------------
    # 2d. Make both DataFrames queryable via SQL
    # --------------------------------------------------------
    # createOrReplaceTempView doesn't move any data — it just gives
    # Spark SQL a name to refer to this in-memory DataFrame by,
    # so the MERGE statements below can reference it.
    good.createOrReplaceTempView("items_batch")
    bad.createOrReplaceTempView("items_bad_batch")

    # --------------------------------------------------------
    # 2e. Write good rows into the real silver table
    # --------------------------------------------------------
    spark_session = batch_df.sparkSession
    spark_session.sql("""
        MERGE INTO azuresalesdatabricks.silver.order_items AS target
        USING items_batch AS source
        ON target.order_item_id = source.order_item_id
        WHEN NOT MATCHED THEN INSERT *
    """)
    # "WHEN NOT MATCHED THEN INSERT" means: only insert rows whose
    # order_item_id doesn't already exist in the target table.
    # This makes the MERGE idempotent — safe to re-run without
    # creating duplicates, even if this batch somehow ran twice.

    # --------------------------------------------------------
    # 2f. Write bad rows into the quarantine table instead
    # --------------------------------------------------------
    spark_session.sql("""
        MERGE INTO azuresalesdatabricks.silver.order_items_quarantine AS target
        USING items_bad_batch AS source
        ON target.order_item_id = source.order_item_id
        WHEN NOT MATCHED THEN INSERT *
    """)
    # Same idempotent pattern, just a different destination table.
    # Nothing gets silently dropped — every row lands somewhere.


# ============================================================
# STEP 3 — Wire the function into the actual streaming query
# ============================================================
query = (bronze_items.writeStream
    .foreachBatch(clean_and_merge_items)   # run our function per batch, instead of a plain append
    .option("checkpointLocation",
            "abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/order_items/")
    # tracks exactly which bronze rows this SPECIFIC stream has
    # already processed — must be a unique path, not shared with
    # any other entity's stream
    .trigger(availableNow=True)   # process everything currently available, then stop
    .start())

# Block here until the stream finishes this run, so you see the
# result before moving on — rather than it running in the background
query.awaitTermination()

## 2.3. Cleaning returns

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window   

# ============================================================
# STEP 1 — Set up the streaming source
# ============================================================
# Same pattern as sales_orders and order_items: read bronze.returns
# as a stream, so only new rows since the last checkpoint show up.
bronze_returns = spark.readStream.table("azuresalesdatabricks.bronze.returns")


# ============================================================
# STEP 2 — Define what happens to EACH micro-batch of new rows
# ============================================================
def clean_and_merge_returns(batch_df, batch_id):

    # --------------------------------------------------------
    # 2a. Light cleaning
    # --------------------------------------------------------
    # return_date arrives as a plain string from CSV (like order_date
    # did) — but unlike sales_orders, our generator never introduced
    # a second date format for returns, so a simple to_date is safe
    # here (no need for try_to_date + coalesce fallback logic).
    cleaned = (batch_df
        .withColumn("return_date", F.to_date("return_date", "yyyy-MM-dd"))
        .withColumn("refund_amount", F.col("refund_amount").cast("double"))
    )

    # --------------------------------------------------------
    # 2b. Dedupe within this batch, just in case
    # --------------------------------------------------------
    # We never deliberately duplicated returns like we did with
    # sales_orders, but this costs nothing and protects us if a
    # file ever gets re-sent or re-processed. Keep the most
    # recently ingested version of each return_id.
    w = Window.partitionBy("return_id").orderBy(F.col("_ingested_at").desc())
    deduped = (cleaned
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn"))

    # --------------------------------------------------------
    # 2c. Make it queryable via SQL for the MERGE step
    # --------------------------------------------------------
    deduped.createOrReplaceTempView("returns_batch")

    # --------------------------------------------------------
    # 2d. Merge into silver — idempotent insert-only
    # --------------------------------------------------------
    # A return, once recorded, doesn't get "updated" in our model —
    # there's no SCD-style change tracking needed here, just:
    # if we've never seen this return_id before, insert it.
    batch_df.sparkSession.sql("""
        MERGE INTO azuresalesdatabricks.silver.returns AS target
        USING returns_batch AS source
        ON target.return_id = source.return_id
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# STEP 3 — Wire the function into the actual streaming query
# ============================================================
query = (bronze_returns.writeStream
    .foreachBatch(clean_and_merge_returns)
    .option("checkpointLocation",
            "abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/returns/")
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

# 3. Cleaning Dimension tables

## 3.1. Cleaning customers table with the SCD2 merge logic itself

In [0]:
# ============================================================
# IMPORTS — run this first, every session
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# PERMANENT — The actual merge function
# ============================================================
# This function is the REAL pipeline logic. It gets defined once,
# then called two different ways:
#   - TODAY: manually, twice, by us, to seed history correctly
#   - FOREVER AFTER: automatically, by Spark, whenever ADF/Autoloader
#     delivers new data — via the writeStream block at the bottom
def merge_customers_scd2(batch_df, batch_id):

    # --------------------------------------------------------
    # Fingerprint the attributes we care about tracking.
    # If ANY of these differ between two versions of the same
    # customer, we treat it as a real, trackable change.
    # --------------------------------------------------------
    hashed = batch_df.withColumn(
        "row_hash",
        F.sha2(F.concat_ws("||", "address", "city", "state",
                            "zip_code", "segment", "email", "phone"), 256)
    )

    # --------------------------------------------------------
    # Within THIS batch only, keep just the latest version of
    # each customer_id — using the source's own updated_at
    # (event time), not our ingestion timestamp.
    # Note: if a customer changes twice within the SAME batch,
    # only the final value survives — this is a known, accepted
    # trade-off for dimension tables (see our earlier discussion).
    # --------------------------------------------------------
    w = Window.partitionBy("customer_id").orderBy(
        F.col("updated_at").desc(), F.col("_ingested_at").desc()
    )
    latest = (hashed
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn"))

    latest.createOrReplaceTempView("customers_latest")
    spark_session = batch_df.sparkSession

    # --------------------------------------------------------
    # MERGE 1 — Close out any CURRENT row whose data changed.
    # Looks for a customer that already has an is_current=true
    # row in the target, but whose hash no longer matches —
    # meaning something about them changed. Mark that row as
    # historical instead of deleting it.
    # --------------------------------------------------------
    spark_session.sql("""
        MERGE INTO azuresalesdatabricks.silver.customers AS target
        USING customers_latest AS source
        ON target.customer_id = source.customer_id AND target.is_current = true
        WHEN MATCHED AND target.row_hash != source.row_hash THEN
          UPDATE SET target.is_current = false,
                     target.effective_end_date = source.updated_at
    """)

    # --------------------------------------------------------
    # MERGE 2 — Insert a fresh CURRENT row for:
    #   (a) brand new customers we've never seen, and
    #   (b) customers whose old row MERGE 1 just closed a
    #       moment ago (they now have no current row, so this
    #       correctly falls into WHEN NOT MATCHED)
    # --------------------------------------------------------
    spark_session.sql("""
        MERGE INTO azuresalesdatabricks.silver.customers AS target
        USING customers_latest AS source
        ON target.customer_id = source.customer_id AND target.is_current = true
        WHEN NOT MATCHED THEN
          INSERT (customer_id, first_name, last_name, email, phone, address, city,
                  state, zip_code, country, segment, row_hash, effective_start_date,
                  effective_end_date, is_current, _source_file, _ingested_at)
          VALUES (source.customer_id, source.first_name, source.last_name, source.email,
                  source.phone, source.address, source.city, source.state, source.zip_code,
                  source.country, source.segment, source.row_hash, source.updated_at,
                  NULL, true, source._source_file, source._ingested_at)
    """)


# ============================================================
# ✅ PERMANENT PIPELINE — this is the ONLY block that survives
# ============================================================
# From tomorrow onward, THIS is what ADF/Databricks will trigger.
# It reads new bronze rows incrementally and calls the SAME
# merge_customers_scd2 function automatically, per micro-batch —
# no manual calls, no steps 2/3, ever again.
bronze_customers = spark.readStream.table("azuresalesdatabricks.bronze.customers")

query = (bronze_customers.writeStream
    .foreachBatch(merge_customers_scd2)
    .option("checkpointLocation",
            "abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/customers/")
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

## 3.2. Cleaning 3 tables(products, stores and sales_reps)

### 3.2.1. The generic, reusable SCD2 merge function

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def make_scd2_merger(target_table, business_key, tracked_columns, event_time_col="_ingested_at"):
    def merge_fn(batch_df, batch_id):
        hashed = batch_df.withColumn(
            "row_hash", F.sha2(F.concat_ws("||", *tracked_columns), 256)
        )

        w = Window.partitionBy(business_key).orderBy(F.col(event_time_col).desc())
        latest = (hashed
            .withColumn("rn", F.row_number().over(w))
            .filter("rn = 1")
            .withColumn("effective_start_date", F.to_date(F.col(event_time_col)))
        )

        # FIX: explicitly select ONLY the columns we actually want to
        # carry into silver — business key, tracked attributes, and our
        # SCD2 bookkeeping columns. This drops bronze-only columns like
        # created_at, _rescued_data, _entity that the target table
        # was never designed to hold.
        select_cols = [business_key] + tracked_columns + ["row_hash", "effective_start_date", "_source_file", "_ingested_at"]
        latest_clean = latest.select(*select_cols)

        latest_clean.createOrReplaceTempView("scd2_batch")
        spark_session = batch_df.sparkSession

        spark_session.sql(f"""
            MERGE INTO {target_table} AS target
            USING scd2_batch AS source
            ON target.{business_key} = source.{business_key} AND target.is_current = true
            WHEN MATCHED AND target.row_hash != source.row_hash THEN
              UPDATE SET target.is_current = false,
                         target.effective_end_date = source.effective_start_date
        """)

        insert_cols = latest_clean.columns   # now guaranteed to match target table's real columns
        col_list = ", ".join(insert_cols)
        val_list = ", ".join(f"source.{c}" for c in insert_cols)

        spark_session.sql(f"""
            MERGE INTO {target_table} AS target
            USING scd2_batch AS source
            ON target.{business_key} = source.{business_key} AND target.is_current = true
            WHEN NOT MATCHED THEN
              INSERT ({col_list}, effective_end_date, is_current)
              VALUES ({val_list}, NULL, true)
        """)
    return merge_fn

### 3.2.2. Configure entities

In [0]:
entities_scd2 = [
    {"entity_name": "products", "target_table": "azuresalesdatabricks.silver.products",
     "business_key": "product_id",
     "tracked_columns": ["product_name", "category", "subcategory", "brand", "unit_cost", "unit_price"], "event_time_col": "created_at"},
    {"entity_name": "stores", "target_table": "azuresalesdatabricks.silver.stores",
     "business_key": "store_id",
     "tracked_columns": ["store_name", "region", "city", "state", "store_type"],
     "event_time_col": "opened_date"},
    {"entity_name": "sales_reps", "target_table": "azuresalesdatabricks.silver.sales_reps",
     "business_key": "rep_id",
     "tracked_columns": ["rep_name", "email", "store_id", "region"],
     "event_time_col": "hire_date"},
]

### 3.2.3. Run each entity

In [0]:
for e in entities_scd2:
    print(f"Running SCD2 merge for {e['entity_name']}...")

    merge_fn = make_scd2_merger(e["target_table"], e["business_key"], e["tracked_columns"], e["event_time_col"])
    bronze_stream = spark.readStream.table(f"azuresalesdatabricks.bronze.{e['entity_name']}")

    query = (bronze_stream.writeStream
        .foreachBatch(merge_fn)
        .option("checkpointLocation", f"abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/{e['entity_name']}/")
        .trigger(availableNow=True)
        .start())
    query.awaitTermination()

    print(f"Done: {e['entity_name']}")